In [ ]:
import shap
from transformers import AutoFeatureExtractor, AutoModel
import torch
import pickle
import numpy as np
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
device = "cpu"
feature_extractor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)
model.eval()
@torch.no_grad()
def extract_feature(image, pool_index=0):
    """Genera le feature facendo CLS-pooling"""
    inputs = feature_extractor(image, return_tensors="pt")
    for k in inputs:
        inputs[k] = inputs[k].to(device)
    outputs = model(**inputs)
    pooled_output = outputs.last_hidden_state[:, pool_index].reshape(-1)
    model.zero_grad()
    del outputs
    del inputs
    return pooled_output

In [ ]:
path_images = "../input/bigImageDataset/imgs"

id2img = pickle.load(open("../embeddings/unique2path.pkl", "rb"))


def openImage(image,fixed_size=(512,512),override_path=None):
    if override_path is not None:
        path=f'{override_path}/{image}'
    else:
        path=f'{path_images}/{image}'
    return Image.open(path).convert('RGBA').convert('RGB').resize((fixed_size[0], fixed_size[1]))

In [ ]:
def merge_images(image1,image2,fixed_size=(512,512)):

    new_image = Image.new("RGB", (fixed_size[0]*2, fixed_size[1]), "black")
    new_image.paste(image1,(0,0))
    new_image.paste(image2,(fixed_size[1],0))
    return new_image

def unmerge_images(image,fixed_size=(512,512)):
    image1=image.crop((0, 0, fixed_size[0], fixed_size[1]))
    image2=image.crop((fixed_size[0], 0, fixed_size[0]*2, fixed_size[1]))
    return image1,image2

In [ ]:
images=list(id2img.items())[:2]
images

In [ ]:
image1=openImage(images[0][1])
image2=openImage(images[1][1])
merged=merge_images(image1,image2)
merged

In [ ]:
SOGLIA=0.5
from IPython.display import display
def batch_predictor_image(images):
    images=[Image.fromarray(np.uint8(i)) for i in images]
    # display(images[0])
    d=torch.nn.CosineSimilarity()
    distances=[]
    with torch.no_grad():
        for images_merged in images:
            image_src,image_trg=unmerge_images(images_merged)
            emb_src=extract_feature(image_src)
            emb_trg=extract_feature(image_trg)
            distance=d(emb_src.unsqueeze(0),emb_trg.unsqueeze(0)).item()
            distance_neg=1-distance
            distance_pos=distance
            distances.append([distance_neg,distance_pos])
    distances=np.array(distances).reshape(-1,2)
    # distances= np.exp(distances)
    # p=torch.from_numpy(distances)
    # distances=torch.nn.functional.softmax(p,1).detach().cpu().numpy()
    return distances


In [ ]:
graph_path="../graphsV2/NFTLevel-Pruning_V2-Weekly2/(2021, 3)/edge_list.csv"
edge_list=pd.read_csv(graph_path)
edge_list=edge_list[edge_list.weight>.7].sort_values(by="weight",ascending=False)
edge_list=edge_list.reset_index(drop=True)
edge_list.head(10)

In [ ]:
edge_index=3
edge=edge_list.iloc[edge_index]
edge=edge_list.sample().iloc[0]
print(edge)
image1=openImage(id2img[edge["source"]])
image2=openImage(id2img[edge["target"]])
merged=merge_images(image1,image2)
merged

In [ ]:
edge

In [ ]:
path_img="/home/user/Immagini/Screenshots"
image1=openImage("5.png",override_path=path_img)
image2=openImage("6.png",override_path=path_img)
merged=merge_images(image1,image2)
print("Logits:",batch_predictor_image([merged]))

merged

In [ ]:
topk = 1
batch_size = 100
n_evals = 1000
merged_as_tensor=np.array(merged)
# define a masker that is used to mask out partitions of the input image.
# masker = shap.maskers.Image("inpaint_telea", merged_as_tensor.shape)
masker = shap.maskers.Image("blur(64,64)", merged_as_tensor.shape)

# create an explainer with model and image masker
explainer = shap.Explainer(batch_predictor_image, masker, output_names=["Not similar","Similar"])
merged_as_tensor.shape

In [ ]:
# feed only one image
shap_values = explainer(merged_as_tensor.reshape(1,512,1024,3), max_evals=n_evals, batch_size=batch_size,
                        outputs=shap.Explanation.argsort.flip[:topk])
shap_values.data = shap_values.data*255
shap_values.values = [val for val in np.moveaxis(shap_values.values[0],-1, 0)]
shap_values.data=shap_values.data.reshape(512,1024,3)

In [ ]:
from shap.plots import colors
def image_(shap_values,
          pixel_values = None,
          labels = None,
          true_labels= None,
          width= 20,
          aspect= 0.2,
          hspace= 0.02,
          labelpad= None,
          cmap = colors.red_transparent_blue,
          show= True):
          
    # support passing an explanation object
    if str(type(shap_values)).endswith("Explanation'>"):
        shap_exp = shap_values
        # feature_names = [shap_exp.feature_names]
        # ind = 0
        if len(shap_exp.output_dims) == 1:
            shap_values = [shap_exp.values[..., i] for i in range(shap_exp.values.shape[-1])]
        elif len(shap_exp.output_dims) == 0:
            shap_values = shap_exp.values
        else:
            raise Exception("Number of outputs needs to have support added!! (probably a simple fix)")
        if pixel_values is None:
            pixel_values = shap_exp.data
        if labels is None:
            labels = shap_exp.output_names

    multi_output = True
    if not isinstance(shap_values, list):
        multi_output = False
        shap_values = [shap_values]

    if len(shap_values[0].shape) == 3:
        shap_values = [v.reshape(1, *v.shape) for v in shap_values]
        pixel_values = pixel_values.reshape(1, *pixel_values.shape)

    # labels: (rows (images) x columns (top_k classes) )
    if labels is not None:
        if isinstance(labels, list):
            labels = np.array(labels).reshape(1, -1)

    label_kwargs = {} if labelpad is None else {'pad': labelpad}

    # plot our explanations
    x = pixel_values
    fig_size = np.array([3 * (len(shap_values) + 1), 2.5 * (x.shape[0] + 1)])
    if fig_size[0] > width:
        fig_size *= width / fig_size[0]
    fig_size=(30,10)    
    fig, axes = plt.subplots(nrows=x.shape[0]+1, ncols=1 , figsize=fig_size)
    if len(axes.shape) == 1:
        axes = axes.reshape(1, axes.size)
    for row in range(x.shape[0]):
        x_curr = x[row].copy()

        # make sure we have a 2D array for grayscale
        if len(x_curr.shape) == 3 and x_curr.shape[2] == 1:
            x_curr = x_curr.reshape(x_curr.shape[:2])

        if len(x_curr.shape) == 3 and x_curr.shape[2] == 3:
            x_curr_gray = (
                    0.2989 * x_curr[:, :, 0] + 0.5870 * x_curr[:, :, 1] + 0.1140 * x_curr[:, :, 2])  # rgb to gray
            x_curr_disp = x_curr

        else:
            x_curr_gray = x_curr
            x_curr_disp = x_curr

        axes[row, 0].imshow(x_curr_disp, cmap=plt.get_cmap('gray'))

        axes[row, 0].axis('off')
        if len(shap_values[0][row].shape) == 2:
            abs_vals = np.stack([np.abs(shap_values[i]) for i in range(len(shap_values))], 0).flatten()
        else:
            abs_vals = np.stack([np.abs(shap_values[i].sum(-1)) for i in range(len(shap_values))], 0).flatten()
        max_val = np.nanpercentile(abs_vals, 99.9)
        for i in range(1):

            sv = shap_values[i][row] if len(shap_values[i][row].shape) == 2 else shap_values[i][row].sum(-1)
            axes[row, i + 1].imshow(x_curr_gray, cmap=plt.get_cmap('gray'), alpha=0.3,
                                    extent=(-1, sv.shape[1], sv.shape[0], -1))
            im = axes[row, i + 1].imshow(sv, cmap=cmap, vmin=-max_val, vmax=max_val)
            axes[row, i + 1].axis('off')
    if hspace == 'auto':
        fig.tight_layout()
    else:
        fig.subplots_adjust(hspace=hspace)

    # if show:
    #     plt.show()

In [ ]:
image_(shap_values=shap_values.values,
                pixel_values=shap_values.data*255,
                labels=shap_values.output_names,
                width=40,
                aspect=.2,
                true_labels=["Original"],
                show=True)

plt.savefig("../graphsV2/plots/5-4.png",bbox_inches='tight')


In [ ]:
shap.image_plot(shap_values=shap_values.values,
                pixel_values=shap_values.data*255,
                labels=shap_values.output_names,
                width=40,
                aspect=.1,
                true_labels=["Original"],
                show=True)
